# Part 1h: Callbacks & TensorBoard
**Author:** Kalhar Mayurbhai Patel (019140511)

Demonstrates Keras callbacks (ModelCheckpoint, TensorBoard, LearningRateScheduler, ReduceLROnPlateau, CSV Logger)
and PyTorch's TensorBoard integration via SummaryWriter.

In [1]:
import numpy as np, os
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## TensorFlow: Various Callbacks

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential, callbacks
import shutil

# Clean previous logs
log_dir = 'logs/tf_housing'
if os.path.exists(log_dir): shutil.rmtree(log_dir)

model = Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Define callbacks
my_callbacks = [
    callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1),
    callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    callbacks.ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    callbacks.CSVLogger('training_log.csv'),
    callbacks.LearningRateScheduler(lambda epoch, lr: lr * 0.995),  # gentle decay
]

history = model.fit(X_train, y_train, epochs=100, batch_size=32,
                    validation_data=(X_test, y_test), callbacks=my_callbacks, verbose=0)

print(f"Final Val MAE: {history.history['val_mae'][-1]:.4f}")
print(f"Training stopped at epoch {len(history.history['loss'])}")
print("\nTensorBoard logs saved to:", log_dir)
print("Run: tensorboard --logdir logs/tf_housing")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Epoch 32: ReduceLROnPlateau reducing learning rate to 0.0004259009729139507.

Epoch 38: ReduceLROnPlateau reducing learning rate to 0.00020664131443481892.

Epoch 47: ReduceLROnPlateau reducing learning rate to 9.876314288703725e-05.

Epoch 53: ReduceLROnPlateau reducing learning rate to 4.7918514610501006e-05.

Epoch 58: ReduceLROnPlateau reducing learning rate to 2.3366235836874694e-05.

Epoch 63: ReduceLROnPlateau reducing learning rate to 1.1393945896998048e-05.
Final Val MAE: 0.3507
Training stopped at epoch 63

TensorBoard logs saved to: logs/tf_housing
Run: tensorboard --logdir logs/tf_housing


In [3]:
# Custom callback example: print every N epochs
class PrintEveryN(tf.keras.callbacks.Callback):
    def __init__(self, n=10):
        self.n = n
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.n == 0:
            print(f"  Epoch {epoch+1}: loss={logs['loss']:.4f}, val_loss={logs['val_loss']:.4f}")

model2 = Sequential([layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
                      layers.Dense(1)])
model2.compile(optimizer='adam', loss='mse')
model2.fit(X_train, y_train, epochs=50, batch_size=32,
           validation_data=(X_test, y_test), callbacks=[PrintEveryN(10)], verbose=0)

  Epoch 10: loss=0.3667, val_loss=0.3736
  Epoch 20: loss=0.3343, val_loss=0.3456
  Epoch 30: loss=0.3127, val_loss=0.3311
  Epoch 40: loss=0.3082, val_loss=0.3195
  Epoch 50: loss=0.3023, val_loss=0.3314


## PyTorch: TensorBoard with SummaryWriter

In [4]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

log_dir_pt = 'logs/pt_housing'
if os.path.exists(log_dir_pt): shutil.rmtree(log_dir_pt)
writer = SummaryWriter(log_dir_pt)

dl = DataLoader(TensorDataset(torch.FloatTensor(X_train).to(device),
    torch.FloatTensor(y_train).unsqueeze(1).to(device)), batch_size=32, shuffle=True)
X_te = torch.FloatTensor(X_test).to(device)
y_te = torch.FloatTensor(y_test).unsqueeze(1).to(device)

model = nn.Sequential(nn.Linear(X_train.shape[1], 64), nn.ReLU(),
                       nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
crit = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)

for ep in range(80):
    model.train()
    epoch_loss = 0
    for xb, yb in dl:
        loss = crit(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        epoch_loss += loss.item()
    model.eval()
    with torch.no_grad():
        val_loss = crit(model(X_te), y_te).item()
    writer.add_scalar('Loss/train', epoch_loss/len(dl), ep)
    writer.add_scalar('Loss/val', val_loss, ep)
    writer.add_scalar('LR', opt.param_groups[0]['lr'], ep)
    # Log weight histograms
    for name, param in model.named_parameters():
        writer.add_histogram(name, param, ep)
    scheduler.step()

writer.close()
print(f"Final Val Loss: {val_loss:.4f}")
print(f"PT TensorBoard logs saved to: {log_dir_pt}")
print("Run: tensorboard --logdir logs/pt_housing")

Final Val Loss: 0.2646
PT TensorBoard logs saved to: logs/pt_housing
Run: tensorboard --logdir logs/pt_housing


## Callbacks Summary
| Callback | Purpose |
|---|---|
| **EarlyStopping** | Stop when val metric stops improving |
| **ModelCheckpoint** | Save the best model weights |
| **TensorBoard** | Visual logging of metrics, histograms, graphs |
| **ReduceLROnPlateau** | Decrease LR when metric plateaus |
| **CSVLogger** | Save training log to a CSV file |
| **LearningRateScheduler** | Custom LR schedule per epoch |
| **Custom Callback** | Any custom logic via on_epoch_end, etc. |